## EDA 

In [1]:
## Imports

import warnings
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
sns.set_theme(style='whitegrid', palette='muted')

TEAL   = '#2EC4B6'
CORAL  = '#E71D36'
GOLD   = '#FF9F1C'
PURPLE = '#8B5CF6'
GREEN  = '#06D6A0'
SLATE  = '#94A3B8'

In [2]:
def skew_label(s):
    if   s >  1: return 'strong right skew → log-transform'
    elif s > 0.5: return 'moderate right skew'
    elif s < -1: return 'strong left skew'
    else:        return 'approx. normal'

In [6]:
categories = ['pan', 'sandal', 'sunscreen', 'wallet']

In [4]:
# Load data
df_pan = pd.read_csv("./data/pan_final.csv")
df_sandal = pd.read_csv("./data/sandal_final.csv")
df_sunscreen = pd.read_csv("./data/sunscreen_final.csv")
df_wallet = pd.read_csv("./data/wallet_final.csv")

In [5]:
# Dataframe Overview
print("Pan dataframe shape:", df_pan.shape)
print("Sandal dataframe shape:", df_sandal.shape)
print("Sunscreen dataframe shape:", df_sunscreen.shape) 
print("Wallet dataframe shape:", df_wallet.shape)

Pan dataframe shape: (4551, 44)
Sandal dataframe shape: (10000, 46)
Sunscreen dataframe shape: (3371, 46)
Wallet dataframe shape: (4151, 52)


In [8]:
for cat in categories:
    df = eval(f"df_{cat}")
    print(f"\n── {cat.upper()} ──")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("Columns:", df.columns.tolist())

    # ── dtype + null summary ──────────────────────────────────────────
    schema = pd.DataFrame({
        'dtype'   : df.dtypes,
        'null_count': df.isnull().sum(),
        'null_pct' : (df.isnull().sum() / len(df) * 100).round(2),
        'nunique'  : df.nunique(),
        'sample'   : df.iloc[0],
    })
    schema = schema.sort_values('null_pct', ascending=False)

    # Flag columns with >5% nulls
    schema['flag'] = schema['null_pct'].apply(lambda x: '⚠ HIGH NULL' if x > 5 else ('· low null' if x > 0 else ''))

    display(schema.style
        .background_gradient(subset=['null_pct'], cmap='Oranges')
        .set_caption(f'Schema & Null Audit — {cat.upper()}')
        .format({'null_pct': '{:.2f}%'})
    )


── PAN ──
Shape: 4,551 rows × 44 columns
Columns: ['id', 'title', 'rating', 'reviews', 'initial_price', 'final_price', 'stock', 'favorite', 'seller_name', 'seller_rating', 'seller_products', 'seller_chats_responded_percentage', 'seller_chat_time_reply', 'seller_joined_date', 'seller_followers', 'sold', 'brand', 'gmv_cal', 'country_of_origin', 'warranty_duration', 'warranty_type', 'material', 'shipped_from', 'Diameter', 'variations_count', 'voucher_status', 'image_count', 'video_count', 'discount', 'pan_type', 'rating_credibility', 'discount_percentage', 'log_reviews', 'log_final_price', 'log_initial_price', 'log_stock', 'log_favorite', 'log_sold', 'log_gmv_cal', 'log_seller_followers', 'log_seller_products', 'log_discount', 'z_sold', 'z_rating_credibility']


,dtype,null_count,null_pct,nunique,sample,flag
id,int64,0,0.00%,4475,29072010882,
title,object,0,0.00%,4379,"Fleco Panci Listrik F-1892t 1.8liter Rice Cooker Mini 450watt / Fleco Multifungtional Electric Rice Cooker | Fleco Rice Cooker F1892t | Penanak Nasi 1,8 Liter Rice Cooker Listrik Multifunsi - SAS",
variations_count,int64,0,0.00%,34,1,
voucher_status,int64,0,0.00%,2,0,
image_count,int64,0,0.00%,18,6,
video_count,int64,0,0.00%,2,0,
discount,float64,0,0.00%,1405,100000.000000,
pan_type,object,0,0.00%,5,Other Cookware,
rating_credibility,float64,0,0.00%,893,0.000000,
discount_percentage,float64,0,0.00%,1544,49.043649,



── SANDAL ──
Shape: 10,000 rows × 46 columns
Columns: ['id', 'title', 'rating', 'reviews', 'initial_price', 'final_price', 'stock', 'favorite', 'seller_name', 'breadcrumb', 'seller_rating', 'seller_products', 'seller_chats_responded_percentage', 'seller_chat_time_reply', 'seller_joined_date', 'seller_followers', 'sold', 'brand', 'gmv_cal', 'shipped_from', 'heel_height', 'material', 'toe_type', 'wide_fit', 'country_of_origin', 'shoe_model', 'fastener_type', 'occasion', 'variations_count', 'voucher_status', 'image_count', 'video_count', 'discount', 'rating_credibility', 'log_sold', 'log_final_price', 'log_seller_products', 'log_seller_followers', 'z_sold', 'z_rating_credibility', 'log_reviews', 'log_initial_price', 'log_stock', 'log_favorite', 'log_gmv_cal', 'discount_percentage']


,dtype,null_count,null_pct,nunique,sample,flag
id,int64,0,0.00%,8286,4638476132,
log_sold,float64,0,0.00%,904,4.204693,
shoe_model,object,0,0.00%,10,Other,
fastener_type,object,0,0.00%,8,Other,
occasion,object,0,0.00%,5,Other,
variations_count,int64,0,0.00%,34,5,
voucher_status,int64,0,0.00%,2,0,
image_count,int64,0,0.00%,20,6,
video_count,int64,0,0.00%,2,0,
discount,float64,0,0.00%,2591,11250.000000,



── SUNSCREEN ──
Shape: 3,371 rows × 46 columns
Columns: ['id', 'title', 'rating', 'reviews', 'initial_price', 'final_price', 'stock', 'favorite', 'seller_name', 'seller_rating', 'seller_products', 'seller_chats_responded_percentage', 'seller_chat_time_reply', 'seller_joined_date', 'seller_followers', 'sold', 'brand', 'flash_sale', 'gmv_cal', 'shipped_from', 'formulation', 'spf', 'country_of_origin', 'variations_count', 'voucher_status', 'image_count', 'video_count', 'discount', 'storage_duration_months', 'volume_ml', 'volume_group', 'spf_num', 'discount_percentage', 'rating_credibility', 'log_reviews', 'log_final_price', 'log_initial_price', 'log_stock', 'log_favorite', 'log_sold', 'log_gmv_cal', 'log_seller_followers', 'log_seller_products', 'log_discount', 'z_sold', 'z_rating_credibility']


,dtype,null_count,null_pct,nunique,sample,flag
volume_ml,float64,2181,64.70%,34,nan,⚠ HIGH NULL
storage_duration_months,float64,424,12.58%,15,24.000000,⚠ HIGH NULL
id,int64,0,0.00%,3342,74170424,
log_reviews,float64,0,0.00%,783,9.682779,
image_count,int64,0,0.00%,21,8,
video_count,int64,0,0.00%,2,0,
discount,float64,0,0.00%,1219,8600.000000,
volume_group,object,0,0.00%,6,unknown,
spf_num,int64,0,0.00%,4,3,
discount_percentage,float64,0,0.00%,1831,17.234469,



── WALLET ──
Shape: 4,151 rows × 52 columns
Columns: ['id', 'title', 'rating', 'reviews', 'initial_price', 'final_price', 'stock', 'favorite', 'seller_name', 'breadcrumb', 'seller_rating', 'seller_products', 'seller_chats_responded_percentage', 'seller_chat_time_reply', 'seller_joined_date', 'seller_followers', 'sold', 'brand', 'gmv_cal', 'motif', 'warranty_type', 'material', 'warranty_duration', 'country_of_origin', 'closure_type', 'card_slots', 'coin_slot', 'leather_texture', 'leather_finish', 'leather_type', 'shipped_from_region', 'variations_count', 'voucher_status', 'image_count', 'video_count', 'discount', 'wallet_type', 'log_reviews', 'log_gmv', 'log_favorite', 'log_sold', 'rating_credibility', 'log_final_price', 'log_initial_price', 'log_stock', 'log_gmv_cal', 'log_seller_followers', 'log_seller_products', 'log_discount', 'discount_percentage', 'z_sold', 'z_rating_credibility']


,dtype,null_count,null_pct,nunique,sample,flag
id,int64,0,0.00%,4151,175110626,
title,object,0,0.00%,4086,BAELLERRY 1063 Dompet Panjang Pria Kulit Long Bifold Wallet WK-SBY,
leather_finish,object,0,0.00%,4,Matte,
leather_type,object,0,0.00%,5,Synthetic Leather,
shipped_from_region,object,0,0.00%,8,Java,
variations_count,int64,0,0.00%,40,4,
voucher_status,int64,0,0.00%,2,0,
image_count,int64,0,0.00%,31,8,
video_count,int64,0,0.00%,2,1,
discount,float64,0,0.00%,1134,56000.000000,


In [ ]:
## IQR Outlier Detection

numeric_cols = ['final_price', 'initial_price', 'reviews', 'sold',
                'stock', 'favorite', 'gmv_cal', 'seller_followers', 'rating']

for cat in categories:
    df = eval(f"df_{cat}")
    outlier_report = []
    for col in numeric_cols:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_out = ((df[col] < lo) | (df[col] > hi)).sum()
        outlier_report.append({
            'column'      : col,
            'Q1'          : round(Q1, 3),
            'Q3'          : round(Q3, 3),
            'IQR_lower'   : round(lo, 3),
            'IQR_upper'   : round(hi, 3),
            'max_value'   : round(df[col].max(), 3),
            'n_outliers'  : n_out,
            'outlier_pct' : round(n_out / len(df) * 100, 3),
        })

    out_df = pd.DataFrame(outlier_report)
    display(out_df.style
        .background_gradient(subset=['outlier_pct'], cmap='Reds')
        .set_caption(f'IQR Outlier Scan — Key Numeric Columns - {cat.upper()}')
        .format({'outlier_pct': '{:.2f}%'})
    )

,column,Q1,Q3,IQR_lower,IQR_upper,max_value,n_outliers,outlier_pct
0,final_price,77286.000000,277000.000000,-222285.000000,576571.000000,9202811.700000,359,7.89%
1,initial_price,98000.000000,379850.000000,-324775.000000,802625.000000,9202811.700000,340,7.47%
2,reviews,5.000000,257.000000,-373.000000,635.000000,60028.000000,192,4.22%
3,sold,6.000000,331.000000,-481.500000,818.500000,76221.000000,228,5.01%
4,stock,0.000000,20.000000,-30.000000,50.000000,1777657.000000,966,21.23%
5,favorite,0.000000,18.000000,-27.000000,45.000000,32093.000000,819,18.00%
6,gmv_cal,0.000000,0.000000,0.000000,0.000000,1999200000.000000,1061,23.31%
7,seller_followers,1376.500000,52226.000000,-74897.800000,128500.200000,4136968.000000,627,13.78%
8,rating,4.600000,5.000000,4.100000,5.500000,5.000000,611,13.43%


,column,Q1,Q3,IQR_lower,IQR_upper,max_value,n_outliers,outlier_pct
0,final_price,2000.000000,161785.500000,-237678.300000,401463.800000,1216249.700000,290,2.90%
1,initial_price,45000.000000,142250.000000,-100875.000000,288125.000000,9999000.000000,642,6.42%
2,reviews,5.000000,114.000000,-158.500000,277.500000,53613.000000,1478,14.78%
3,sold,10.000000,237.000000,-330.500000,577.500000,9900.000000,1447,14.47%
4,stock,9.000000,1177.000000,-1743.000000,2929.000000,159988580.000000,1968,19.68%
5,favorite,0.000000,212.100000,-318.200000,530.300000,21563.500000,402,4.02%
6,gmv_cal,479200.000000,9944810.000000,-13719215.000000,24143225.000000,1359320000.000000,1485,14.85%
7,seller_followers,0.000000,204864.500000,-307296.700000,512161.100000,439895.100000,0,0.00%
8,rating,4.800000,5.000000,4.400000,5.300000,5.000000,566,5.66%


,column,Q1,Q3,IQR_lower,IQR_upper,max_value,n_outliers,outlier_pct
0,final_price,30900.000000,59000.000000,-11250.000000,101150.000000,1259600.000000,434,12.87%
1,initial_price,40000.000000,89000.000000,-33500.000000,162500.000000,2998000.000000,400,11.87%
2,reviews,3.000000,138.000000,-199.500000,340.500000,252721.000000,553,16.40%
3,sold,8.000000,409.000000,-593.500000,1010.500000,326378.000000,533,15.81%
4,stock,3.000000,266.000000,-391.500000,660.500000,70039927.000000,619,18.36%
5,favorite,1.000000,46.000000,-66.500000,113.500000,86449.000000,503,14.92%
6,gmv_cal,182250.000000,8681500.000000,-12566625.000000,21430375.000000,3477300000.000000,527,15.63%
7,seller_followers,1800.000000,37400.000000,-51600.000000,90800.000000,8156548.000000,473,14.03%
8,rating,4.800000,5.000000,4.500000,5.300000,5.000000,167,4.95%


,column,Q1,Q3,IQR_lower,IQR_upper,max_value,n_outliers,outlier_pct
0,final_price,52000.000000,89900.000000,-4850.000000,146750.000000,600000.000000,496,11.95%
1,initial_price,85000.000000,175000.000000,-50000.000000,310000.000000,1339000.000000,185,4.46%
2,reviews,5.000000,157.500000,-223.800000,386.200000,7836.500000,647,15.59%
3,sold,11.000000,373.500000,-532.800000,917.200000,9450.000000,634,15.27%
4,stock,10.000000,705.500000,-1033.200000,1748.800000,24899.000000,764,18.41%
5,favorite,6.000000,111.000000,-151.500000,268.500000,997.000000,584,14.07%
6,gmv_cal,764875.000000,26411300.000000,-37704762.500000,64880937.500000,730251000.000000,572,13.78%
7,seller_followers,4100.000000,232600.000000,-338650.000000,575350.000000,1266960.000000,333,8.02%
8,rating,4.800000,5.000000,4.500000,5.300000,5.000000,170,4.10%
